# ⚙️ Feature Engineering

## Enterprise AutoML Platform

## 📖 About this Notebook

This notebook demonstrates feature engineering techniques including feature creation, encoding, scaling, variance filtering, and mutual information-based feature selection.

These transformations improve model performance and data quality.

Sections

1. Title
2. Objective
3. Import Libraries
4. Load Dataset
5. Dataset Overview
6. Missing Value Handling
7. Feature Creation
8. Encoding
9. Scaling
10. Variance Threshold
11. Mutual Information
12. Feature Selection
13. Selected Features
14. Save Engineered Dataset
15. Conclusion


### Objective

The objective of this notebook is to transform raw features into machine-learning-ready features.

This notebook covers:

- Missing Value Handling
- Feature Creation
- Encoding
- Scaling
- Variance Threshold
- Mutual Information
- Feature Selection

Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif,
)

from sklearn.model_selection import train_test_split

Load Dataset

In [ ]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

Dataset Shape

In [ ]:
df.shape

Missing Values

In [ ]:
df.isnull().sum()

Target Encoding

In [ ]:
df["Churn"] = df["Churn"].map(
    {
        "No":0,
        "Yes":1,
    }
)

Feature Creation

In [ ]:
df["ChargesPerMonth"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
)

Split Features

In [ ]:
X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

Numerical Columns

In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

Categorical Columns

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object","category","bool"]
).columns.tolist()

Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

Numerical Pipeline

In [ ]:
numerical_pipeline = Pipeline(
    [
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

Categorical Pipeline

In [ ]:
categorical_pipeline = Pipeline(
    [
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            numerical_pipeline,
            numerical_columns,
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_columns,
        ),
    ]
)

Transform Dataset

In [ ]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

Feature Names

In [ ]:
feature_names = preprocessor.get_feature_names_out()

feature_names[:10]

Convert to DataFrame

In [ ]:
X_train = pd.DataFrame(
    X_train,
    columns=feature_names,
)

Variance Threshold

In [ ]:
selector = VarianceThreshold()

X_variance = selector.fit_transform(
    X_train
)

selected_columns = X_train.columns[
    selector.get_support()
]

print("Selected Features :",len(selected_columns))

Mutual Information

In [ ]:
selector = SelectKBest(
    score_func=mutual_info_classif,
    k=20,
)

selector.fit(
    X_train,
    y_train,
)

scores = pd.DataFrame(
    {
        "Feature":X_train.columns,
        "Score":selector.scores_,
    }
)

scores.sort_values(
    by="Score",
    ascending=False,
).head(20)

Top Features

In [ ]:
top_features = scores.sort_values(
    by="Score",
    ascending=False,
).head(20)

top_features

Feature Importance Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.barh(
    top_features["Feature"],
    top_features["Score"],
)

plt.gca().invert_yaxis()

plt.title("Top Features")

plt.show()

Save Engineered Dataset

In [ ]:
engineered = X_train.copy()

engineered["Target"] = y_train.values

engineered.to_csv(
    "../data/engineered_dataset.csv",
    index=False,
)

Conclusion

# ✅ Conclusion

Completed:

- Missing Value Handling
- Feature Creation
- Encoding
- Scaling
- Variance Threshold
- Mutual Information
- Feature Selection

The engineered dataset is now ready for model training.

---

# 🏭 Production Implementation

The Enterprise AutoML Platform performs feature engineering using dedicated reusable components.

Implemented modules include:

- Feature Engineering
- Feature Selection
- Variance Threshold
- Mutual Information
- Automatic Feature Processing

These components are integrated into the production training pipeline.